# Reproducing Pakhomov et al. (2023) from dandiset 001027

Paper. Pakhomov AG, Gudvangen E, Mangalanathan U, Kondratiev O, Redondo L, Semenov I (2023).
Next generation CANCAN focusing for remote stimulation by nanosecond electric pulses.
*Bioelectrochemistry* 152:108437. [doi:10.1016/j.bioelechem.2023.108437](https://doi.org/10.1016/j.bioelechem.2023.108437).
The claims file was written from the accepted author manuscript in PubMed Central (PMC10247516),
with its Methods and figure legends; the version of record is not open access, and there is no
supplement or linked code.

Dandiset. [DANDI:001027](https://dandiarchive.org/dandiset/001027/0.240605.1859), version
0.240605.1859. The paper cites no version of the dandiset. This is the only published version, and
the draft has the same ten assets and size (3,667,103,151 bytes) and was last modified when this
version was published, so the most recent published version is pinned.

What the dandiset holds. Ten NWB files, one dish each, with one stitched 10x YO-PRO-1 image of a
CHO monolayer after a pulse protocol in a quadrupole array: Packet 1 (3 dishes), Packet 2 (4) and
Packet 3 (3), each rotated once from every electrode, first pulse 4 kV. This is the experiment of
Figure 8. The single-cell time-lapse of Figure 4 and the monolayer experiments of Figures 5, 6, 7
and 9 are not in it.

| Claim | Tested here |
|---|---|
| fig4-unipolar-strongest, fig4-bipolar-weaker, fig4-sham-no-uptake, fig4-twenty-fold-cancellation | Blocked, data missing (single-cell time-lapse not deposited) |
| fig5a-trains-stronger-throughout, fig5a-packet-center-peak-e1-stronger, fig5bc-rotation-focuses-center | Blocked, data missing |
| fig6a-more-packets-more-uptake, fig6b-center-to-periphery-by-number, fig6-peak-narrower | Blocked, data missing |
| fig7a-amplitude-scales-uniformly, fig7bd-targeting-independent-of-amplitude, array-size-independence | Blocked, data missing |
| fig8a-equal-center-uptake | Tested |
| fig8a-more-uptake-near-electrodes | Tested |
| fig8b-packet2-best-targeting | Tested |
| fig9-gap-fades-periphery-faster, fig9b-gap-boosts-targeting | Blocked, data missing |

The helper module `cancan.py` streams the files, finds the electrodes and measures the ROI
profiles; its docstring says where each parameter comes from.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import cancan

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
dishes = cancan.load_all()
for d in dishes:
    d["electrodes"] = cancan.find_electrodes(d["img"], d["valid"])
    d["background"], d["background_each"] = cancan.background(d["img"], d["electrodes"])
print(f"loaded {len(dishes)} files from dandiset {cancan.DANDISET_ID} version {cancan.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

What was loaded, and each check of the metadata against the values. The files hold one image
each and no time series, so there is no trial table or clock to align; the check that the data
streams share a clock does not apply.

In [ ]:
info = pd.DataFrame([{
    "file": d["path"], "subject_id": d["subject_id"], "packet": d["packet"], "first_pulse_kV": d["kv"],
    "session_start": d["session_start"][:16], "shape": d["shape"], "dtype": d["dtype"],
    "max_value": d["max_value"], "unimaged_frac": round(d["zero_frac"], 3), "sha256": d["sha256"][:12]}
    for d in dishes]).sort_values(["packet", "session_start"])
print(info.to_string(index=False))
print("\nDishes per packet (paper, Fig. 8 legend: n = 3-4):")
print(info.packet.value_counts().sort_index().to_string())
print("\nFirst-pulse amplitudes in the descriptions (kV):", sorted(info.first_pulse_kV.unique()))
print("Distinct SHA-256 digests:", info.sha256.nunique(), "of", len(info))
print("\nPacket wording in the session descriptions:")
for p in (1, 2, 3):
    desc = next(d["description"] for d in dishes if d["packet"] == p)
    print(f"  Protocol{p}:", desc[desc.find(f"Protocol{p}"):desc.find("Detailed")].strip())
print("\nOne of the session descriptions in full:\n ", dishes[0]["description"])

Every description names a first pulse of 4.0 kV and one rotation (one packet from each
electrode), as the Figure 8 legend says, and the packet counts (3, 4, 3) fit its n = 3-4. The
descriptions define Protocol 1 as 4 phases, Protocol 2 as 8 phases without delay and Protocol 3 as
8 phases with 5-us delays, as the paper defines Packets 1, 2 and 3 (Section 2.3).

In [ ]:
meta = pd.DataFrame([{
    "subject_id": d["subject_id"], "unit": d["unit"], "resolution": d["resolution"], "conversion": d["conversion"],
    "species": d["species"].split("/")[-1], "strain": d["strain"], "age": d["age"],
    "device": d["device"].split("Additional metadata: ")[1]} for d in dishes])
print(meta.to_string(index=False))

# Are any two images the same recording? Compare the reduced images on a common 64-fold grid.
small = {d["subject_id"]: d["img"].reshape(160, 16, 160, 16).mean(axis=(1, 3)).ravel() for d in dishes}
ids = list(small)
corr = pd.DataFrame(np.corrcoef([small[i] for i in ids]), index=ids, columns=ids)
off = corr.where(~np.eye(len(ids), dtype=bool))
print("\nLargest correlation between two different dishes' images:", round(float(np.nanmax(off.values)), 3))
print("Any two reduced images identical:", any(np.array_equal(dishes[i]["img"], dishes[j]["img"])
                                              for i in range(len(dishes)) for j in range(i + 1, len(dishes))))

The ten files are distinct recordings: no two digests or images are identical (the largest
correlation between two dishes' reduced images is 0.86). The image unit is "n.a." and the
`resolution` field is 96 in nine files and 1.0 in dish 10, neither a pixel size, so the files do
not give the scale of the image. The subject is a dish of the CHO cell line, and the species identifier
NCBITaxon 10029 is *Cricetulus griseus*, the Chinese hamster; the dandiset's summary names it as
"Cricetulus griseus - Cricetulus aureus". The age "P1D" has no meaning for a cell line. Nine files
name the ORCA-Flash4.0 camera and a 10x objective (Methods: "10x, 0.38 NA"; the files say NA 0.40);
dish 10 names no device, and its session starts on 18 March at 18:50, the evening after the other
Packet 2 and 3 dishes.

In [ ]:
print("Rows (pixels) at which the left edge of the imaged area steps (the stitching tile grid):")
for d in dishes[:3]:
    st = cancan.tile_step(d["valid"])
    print(f"  dish {d['subject_id']}: {st}; differences {np.diff(st).tolist()}")
print("ORCA-Flash4 sensor: 2048 x 2048 pixels of 6.5 um; at 10x, 0.65 um per pixel.")
print("Assumed scale:", cancan.UM_PER_PX, "um per pixel; one block of the reduced image =",
      cancan.BLOCK * cancan.UM_PER_PX, "um")

geo = []
for d in dishes:
    g = cancan.geometry(d["electrodes"])
    geo.append({"dish": d["subject_id"], "packet": d["packet"],
                "diag_cc_mm": np.round(g["diag_cc_mm"], 2).tolist(),
                "diag_edge_mm": np.round(g["diag_edge_mm"], 2).tolist(),
                "side_cc_mm": np.round(g["side_cc_mm"], 2).tolist(),
                "footprint_diameter_mm": np.round(g["disk_diameter_mm"], 2).tolist(),
                "half_edge_minus_5pct_mm": round(float(np.mean(g["diag_edge_mm"]) * (0.5 - cancan.EXCLUDE_FRAC)), 2)})
geo = pd.DataFrame(geo).sort_values(["packet", "dish"])
print("\nElectrode geometry found in each image (mm at the assumed scale):")
print(geo.to_string(index=False))
allg = [cancan.geometry(d["electrodes"]) for d in dishes]
cc = np.concatenate([g["diag_cc_mm"] for g in allg]); ee = np.concatenate([g["diag_edge_mm"] for g in allg])
fp = np.concatenate([g["disk_diameter_mm"] for g in allg])
print(f"\nCenter-to-center diagonal {cc.min():.2f} to {cc.max():.2f} mm; edge-to-edge {ee.min():.2f} to {ee.max():.2f} mm; "
      f"footprints {fp.min():.2f} to {fp.max():.2f} mm across (needles 1.5 mm, Methods 2.3).")
print("Paper: ROIs cover +/-1.65 mm (small array) or +/-3.65 mm (large array, 8.4-mm diagonal), "
      "excluding <5% of the diagonal next to the electrodes.")

The image is stitched from tiles that repeat every 1840 to 1850 pixels (dish 10's grid differs in one step), about 90% of the camera's
2048, which is consistent with unbinned tiles overlapping by 10% and not resampled, and so with
0.65 um per pixel. At that scale the electrode footprints are 1.7 to 2.0 mm across, a little wider
than the 1.5-mm needles, as a crushed or unexposed rim would make them. The arrays are the small one:
the edge-to-edge diagonal is about 3.5 mm, and 5% short of each electrode leaves about 1.6 mm from
the center, which is the +/-1.65 mm span the Methods give for the small array. The large array
(8.4-mm diagonal) would need a scale about 2.4 times larger, and its footprints would then be about
4.5 mm across, three times the needle. The scale is consistent with the data but is not stated
in them, so the ROIs are also placed by a rule that needs no scale (`rel`, below).

In [ ]:
bgt = pd.DataFrame([{"dish": d["subject_id"], "packet": d["packet"], "date": d["session_start"][:10],
                     "background_mean": round(d["background"], 1),
                     "background_per_electrode": np.round(d["background_each"], 1).tolist()} for d in dishes])
print("Median intensity inside each electrode footprint (central 60% of the radius), raw counts:")
print(bgt.sort_values(["packet", "dish"]).to_string(index=False))
print("\nBy imaging date:")
print(bgt.groupby("date").background_mean.agg(["count", "mean", "min", "max"]).round(1).to_string())

# Saturation within the ROI strip of each dish (the main ROI geometry), from the per-block fraction.
rows = []
for d in dishes:
    e = d["electrodes"]; c = np.array([(a, b) for a, b, _ in e]); cen = c.mean(0)
    yy, xx = np.mgrid[:d["img"].shape[0], :d["img"].shape[1]]
    half = cancan.HALF_SPAN_MM * 1000 / cancan.UM_PER_PX / cancan.BLOCK
    mask = np.zeros(d["img"].shape, bool)
    for i, j in [(0, 2), (1, 3)]:
        u = (c[j] - c[i]) / np.hypot(*(c[j] - c[i])); nrm = np.array([-u[1], u[0]])
        along = (yy - cen[0]) * u[0] + (xx - cen[1]) * u[1]
        across = (yy - cen[0]) * nrm[0] + (xx - cen[1]) * nrm[1]
        mask |= (np.abs(along) <= half) & (np.abs(across) <= half / 46)
    rows.append({"dish": d["subject_id"], "saturated_frac_in_rois": float(d["saturated"][mask].mean()),
                 "saturated_frac_image": float(d["saturated"].mean())})
sat = pd.DataFrame(rows)
print("\nFraction of pixels >= 65000 (of 65535):")
print(sat.to_string(index=False, float_format=lambda v: f"{v:.2e}"))

The footprints' interior is the only region of each image where cells were not exposed, and the
files hold no sham dish or cell-free region, so it serves as the background in the
background-subtracted reading. The intensity there is about twice as high in the three Packet 1
dishes (imaged on 24 March 2021) as in the seven others (17 and 18 March), so the a.u. scale
differs between imaging days. The files do not record exposure or gain. Saturated pixels are
too few to affect an ROI mean.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8.5))
order = sorted(dishes, key=lambda d: (d["packet"], int(d["subject_id"])))
for ax, d in zip(axes.flat, order):
    img = d["img"]
    vmax = np.percentile(img[d["valid"]], 99.5)
    ax.imshow(img, cmap="gray", vmin=0, vmax=vmax)
    e = d["electrodes"]; c = np.array([(a, b) for a, b, _ in e]); cen = c.mean(0)
    for k, (cy, cx, r) in enumerate(e):
        ax.add_patch(plt.Circle((cx, cy), r, fill=False, color="tab:red", lw=1))
        ax.text(cx, cy, f"E{k}", color="yellow", ha="center", va="center", fontsize=8)
    half = cancan.HALF_SPAN_MM * 1000 / cancan.UM_PER_PX / cancan.BLOCK
    for i, j, col in [(0, 2, "cyan"), (1, 3, "orange")]:
        u = (c[j] - c[i]) / np.hypot(*(c[j] - c[i]))
        a, b = cen - u * half, cen + u * half
        ax.plot([a[1], b[1]], [a[0], b[0]], color=col, lw=1)
    ax.set_xlim(cen[1] - 900, cen[1] + 900); ax.set_ylim(cen[0] + 900, cen[0] - 900)
    ax.set_title(f"dish {d['subject_id']}, Protocol {d['packet']} (0 to {vmax:.0f})", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Electrode footprints found (red) and the ROI span of each diagonal (cyan d1: E0-E2, orange d2: E1-E3); "
             "grey scale 0 to the 99.5th percentile, shown in each title")
plt.tight_layout(); plt.show()
print("Each panel spans 1800 x 1800 blocks around the array center; every electrode lies inside it.")

## Profiles under every reading

The paper gives the span and number of ROIs but leaves open the details below, which the claims
file names or which the Methods leave open (author, step 1.6). Each is run both or all ways:

- `geom`: ROIs cover +/-1.65 mm from the geometrical center at 0.65 um per pixel (`abs`; Methods
  2.4), or run to 5% of the edge-to-edge diagonal short of each electrode footprint (`rel`; Methods
  2.4, "<5% of the diagonal length"), which needs no scale.
- `n_per_diag`: "92 ... ROI, placed along one or both diagonals" (Methods 2.4) read as 46 per
  diagonal (92 over both) or 92 per diagonal.
- `center`: the mean of the two ROIs flanking 0, or one ROI centered at 0 (claims file).
- `diag`: the mean of both diagonals, or one (d1, E0 to E2; d2, E1 to E3). Which electrode is e1 is
  not in the files, so d1 and d2 are both run for "one diagonal".
- `bg`: intensities as recorded, or less the per-dish background inside the electrode footprints.
  The Methods say nothing of a background for monolayers (Section 2.2 subtracts one for single cells).

The main reading, used for the measured values, is `abs`, 46 per diagonal, `flank`, `both`,
`raw`: the stated span and ROI count, both diagonals as Figure 5B averages them, and the a.u. values
as recorded, as the claims file says they are judged.

In [ ]:
rows, profs = [], []
for d in sorted(dishes, key=lambda d: (d["packet"], int(d["subject_id"]))):
    r, p = cancan.dish_readings(d)
    rows += r; profs.append((d["subject_id"], d["packet"], d["background"], p))
R = pd.DataFrame(rows)
R["pct"] = 100 * R.center_au / R.periph_au
keys = ["geom", "n_per_diag", "center", "bg", "diag"]
print(len(R), "rows:", R.dish.nunique(), "dishes x", len(R) // R.dish.nunique(), "readings")
main = (R.geom == "abs") & (R.n_per_diag == 46) & (R.center == "flank") & (R.bg == "raw") & (R.diag == "both")
print("\nMain reading, per dish:")
print(R[main][["dish", "packet", "center_au", "periph_au", "periph_min_end_au", "periph_max_end_au", "pct"]]
      .round(1).to_string(index=False))
print("\nMain reading, background subtracted, per dish:")
mainbg = (R.geom == "abs") & (R.n_per_diag == 46) & (R.center == "flank") & (R.bg == "bgsub") & (R.diag == "both")
print(R[mainbg][["dish", "packet", "center_au", "periph_au", "pct"]].round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:green"}
ranges = []
for ax, sub in zip(axes, [False, True]):
    for p in (1, 2, 3):
        ps = [x for x in profs if x[1] == p]
        t, prof_ = cancan.mean_profile([x[3] for x in ps], ("abs", 46, "flank"),
                                       bg=[x[2] for x in ps] if sub else None)
        m, se = prof_.mean(0), prof_.std(0, ddof=1) / np.sqrt(len(ps))
        ax.errorbar(t, m, se, color=colors[p], label=f"Protocol {p} (n={len(ps)})", capsize=2, lw=1)
        ranges.append((sub, p, float((m - se).min()), float((m + se).max())))
    ax.set_xlabel("distance from the array center along the diagonal (mm)")
    ax.set_ylabel("YP emission (a.u.)" + (", less background" if sub else ", raw"))
    ax.legend(); ax.set_title("mean +/- s.e. over dishes, mean of both diagonals")
plt.tight_layout(); plt.show()
for sub, p, lo, hi in ranges:
    print(f"{'background subtracted' if sub else 'raw':>22}, Protocol {p}: plotted range {lo:.0f} to {hi:.0f} a.u.; no points outside the axes")

## fig8a-equal-center-uptake

> Packets 1, 2 and 3 (4 kV, 1Px4) produce the same YP uptake in the center of the array, 215 to 235 a.u.
> (Results, 3.4, paragraph 2; Figure 8A)

Criterion: the mean over dishes of the center ROI per packet, in a.u., each within 193.5 to
258.5 a.u.; no pair of packets differs by a two-sided Student's t-test at p < 0.05. Reproduced when
both hold; partially reproduced when no pair differs but a value is out of range.

In [ ]:
def ttest(a, b):
    return stats.ttest_ind(a, b, equal_var=True)

def center_outcome(g, labels):
    vals = {p: g[g.packet == q].center_au.values for p, q in labels.items()}
    means = {p: v.mean() for p, v in vals.items()}
    in_range = {p: 193.5 <= m <= 258.5 for p, m in means.items()}
    ps = {f"{a}v{b}": ttest(vals[a], vals[b]).pvalue for a, b in [(1, 2), (1, 3), (2, 3)]}
    differ = any(p < 0.05 for p in ps.values())
    out = "reproduced" if all(in_range.values()) and not differ else (
        "partially_reproduced" if not differ else "not_reproduced")
    return {**{f"P{p}_center": round(m, 1) for p, m in means.items()},
            **{f"p_{k}": float(f"{v:.3g}") for k, v in ps.items()}, "outcome": out}

AS_GIVEN = {1: 1, 2: 2, 3: 3}
SWAPPED = {1: 1, 2: 3, 3: 2}   # variant: Packet 2 read from the Protocol 3 files and the reverse
c_tab = R.groupby(keys).apply(lambda g: pd.Series(center_outcome(g, AS_GIVEN))).reset_index()
main_row = c_tab[(c_tab.geom == "abs") & (c_tab.n_per_diag == 46) & (c_tab.center == "flank") & (c_tab.bg == "raw") & (c_tab.diag == "both")]
print("Main reading:"); print(main_row.to_string(index=False))
print("\nOutcomes over all", len(c_tab), "readings:"); print(c_tab.outcome.value_counts().to_string())
print("\nBy background reading:"); print(c_tab.groupby("bg").outcome.value_counts().to_string())
print("\nRange of each packet's center mean over readings (a.u.):")
for p in (1, 2, 3):
    for b in ("raw", "bgsub"):
        v = c_tab[c_tab.bg == b][f"P{p}_center"]
        print(f"  Packet {p}, {b}: {v.min():.1f} to {v.max():.1f}")
print("\nLargest p-value of each pair over readings:", {k: c_tab[k].max() for k in ["p_1v2", "p_1v3", "p_2v3"]})
print("Smallest p-value of each pair over readings:", {k: c_tab[k].min() for k in ["p_1v2", "p_1v3", "p_2v3"]})

In [ ]:
print("Per-dish center values, main reading, raw and background subtracted (a.u.):")
print(R[(main | mainbg)].pivot_table(index=["packet", "dish"], columns="bg", values="center_au").round(1).to_string())
print("\nThe same, as a ratio to each dish's own background (center / background):")
cb = R[main].copy(); cb["bg_level"] = [next(d["background"] for d in dishes if d["subject_id"] == s) for s in cb.dish]
cb["center_over_bg"] = cb.center_au / cb.bg_level
print(cb[["dish", "packet", "center_over_bg"]].round(2).to_string(index=False))
for p in (1, 2, 3):
    v = cb[cb.packet == p].center_over_bg
    print(f"  Packet {p}: mean {v.mean():.2f} +/- {v.std(ddof=1)/np.sqrt(len(v)):.2f} s.e.")
print("p (center/background), 1 v 2:", round(ttest(cb[cb.packet == 1].center_over_bg, cb[cb.packet == 2].center_over_bg).pvalue, 4),
      "; 1 v 3:", round(ttest(cb[cb.packet == 1].center_over_bg, cb[cb.packet == 3].center_over_bg).pvalue, 4),
      "; 2 v 3:", round(ttest(cb[cb.packet == 2].center_over_bg, cb[cb.packet == 3].center_over_bg).pvalue, 4))

**Result: not reproduced.** Under the main reading the center means are 648.7 a.u. (Packet 1), 388.5
(Packet 2) and 383.4 (Packet 3), none within 193.5 to 258.5, and Packet 1 differs from Packets 2 and 3
(p = 0.00044 and 0.00093). All 48 readings give not reproduced: Packet 1 is 332.5 to 381.1 a.u. even
with the background subtracted, and its p-values against the other two stay below 0.05 (largest
0.047 and 0.026). Packets 2 and 3 do not differ in any reading (smallest p 0.174); with the
background subtracted they are 213.4 to 225.2 and 171.5 to 232.8 a.u. The Packet 1 dishes were
imaged on another day with a background about twice as high (mean 308.1 a.u., against 161.8 for
the dishes of 17 March and 150.3 for that of 18 March); as a ratio
to each dish's own background the centers are 2.11, 2.37 and 2.50, and only Packet 1 against
Packet 3 differs (p = 0.047). That ratio is not the paper's measure and does not decide the outcome.

## fig8a-more-uptake-near-electrodes

> Packets 1 and 3 cause much more YP uptake near the electrodes (250 to 350 a.u.) than Packet 2
> (about 120 a.u.), p < 0.01. (Results, 3.4, paragraph 2; Figure 8A)

Criterion: near the electrodes is the most peripheral ROI; readings (a) the mean of the ROIs
closest to each electrode, (b) the ROI closest to e1; one diagonal or both. Reproduced when, under
one reading, Packets 1 and 3 lie in 225 to 385 a.u., Packet 2 in 108 to 132 a.u., and each of
Packets 1 and 3 exceeds Packet 2 by a two-sided Student's t-test at p < 0.01; partially reproduced
when the tests hold and the values do not. Reading (b) cannot be run: the files do not say which
electrode is e1 (the end values of each electrode are shown below for information).

In [ ]:
def near_outcome(g, labels):
    vals = {p: g[g.packet == q].periph_au.values for p, q in labels.items()}
    means = {p: v.mean() for p, v in vals.items()}
    rng = {1: (225, 385), 2: (108, 132), 3: (225, 385)}
    in_range = all(rng[p][0] <= means[p] <= rng[p][1] for p in (1, 2, 3))
    t13 = {p: ttest(vals[p], vals[2]) for p in (1, 3)}
    tests = all(t13[p].pvalue < 0.01 and t13[p].statistic > 0 for p in (1, 3))
    out = "reproduced" if (in_range and tests) else ("partially_reproduced" if tests else "not_reproduced")
    return {**{f"P{p}_periph": round(m, 1) for p, m in means.items()},
            "t_1v2": round(t13[1].statistic, 2), "p_1v2": float(f"{t13[1].pvalue:.3g}"),
            "t_3v2": round(t13[3].statistic, 2), "p_3v2": float(f"{t13[3].pvalue:.3g}"), "outcome": out}

n_tab = R.groupby(keys).apply(lambda g: pd.Series(near_outcome(g, AS_GIVEN))).reset_index()
print("Main reading:")
print(n_tab[(n_tab.geom == "abs") & (n_tab.n_per_diag == 46) & (n_tab.center == "flank") & (n_tab.bg == "raw") & (n_tab.diag == "both")].to_string(index=False))
print("\nOutcomes over all", len(n_tab), "readings:"); print(n_tab.outcome.value_counts().to_string())
print("\nReadings in which Packet 3 is above Packet 2 near the electrodes:", int((n_tab.P3_periph > n_tab.P2_periph).sum()), "of", len(n_tab))
print("Range over readings (a.u.):", {f"P{p}": (n_tab[f'P{p}_periph'].min(), n_tab[f'P{p}_periph'].max()) for p in (1, 2, 3)})
print("t (Packet 3 minus Packet 2) over readings:", n_tab.t_3v2.min(), "to", n_tab.t_3v2.max())
print("t (Packet 1 minus Packet 2) over readings:", n_tab.t_1v2.min(), "to", n_tab.t_1v2.max(), "; p", n_tab.p_1v2.min(), "to", n_tab.p_1v2.max())

In [ ]:
ends = []
for sid, p, bgv, pr in profs:
    sv = cancan.summary_values(pr[("abs", 46, "flank")])
    ends.append({"dish": sid, "packet": p, **{f"E{k}": round(sv[f"end_e{k}"], 1) for k in range(4)}})
ends = pd.DataFrame(ends)
print("Most peripheral ROI next to each electrode (main geometry, raw a.u.); E0 top left, then clockwise in the image:")
print(ends.to_string(index=False))
print("\nPer-packet mean of each electrode's end value:")
print(ends.groupby("packet")[["E0", "E1", "E2", "E3"]].mean().round(1).to_string())

**Result: not reproduced.** Near the electrodes the Protocol 3 files show the least uptake, where
the claim has Packet 3 above Packet 2: under the main reading 806.9 a.u. (Packet 1), 480.9 (Packet 2)
and 296.2 (Packet 3), with Packet 3 below Packet 2 (t = -3.88, p = 0.012). Packet 3 is below
Packet 2 in all 48 readings (t from -5.3 to -1.84). Packet 1 exceeds Packet 2 in every reading
(t from 1.61 to 10.35, p from 0.00015 to 0.168). Reading (b), the ROI next to e1, could not be run,
as the files do not say which electrode is e1; next to each of the four electrodes the Packet 3
mean is below the Packet 2 mean (238.8 against 439.7, 317.6 against 579.6, 270.2 against 446.7 and
358.1 against 457.4 a.u.), so no choice of e1 would reverse this.

## fig8b-packet2-best-targeting

> Normalized to the most peripheral ROI, YP emission in the center is 183% for Packet 2 and drops to
> 83% for Packet 3 and 74% for Packet 1. (Results, 3.4, paragraph 2; Figure 8B)

Criterion: normalization per profile to its most peripheral ROI; readings (a) normalize each dish
then average, (b) normalize the mean profile; peripheral ROI and diagonals as in the previous claim.
Reproduced when all three values are within tolerance (164.7 to 201.3%, 74.7 to 91.3%, 66.6 to
81.4%) under one reading; partially reproduced when Packet 2 is the highest and above 100% while
Packets 1 and 3 are below 100%, without all values in tolerance.

In [ ]:
TOL = {2: (164.7, 201.3), 3: (74.7, 91.3), 1: (66.6, 81.4)}

def pct_outcome(g, labels, norm):
    v = {}
    for p, q in labels.items():
        s = g[g.packet == q]
        v[p] = s.pct.mean() if norm == "per_dish" else 100 * s.center_au.mean() / s.periph_au.mean()
    allin = all(TOL[p][0] <= v[p] <= TOL[p][1] for p in (1, 2, 3))
    partial = v[2] > max(v[1], v[3]) and v[2] > 100 and v[1] < 100 and v[3] < 100
    out = "reproduced" if allin else ("partially_reproduced" if partial else "not_reproduced")
    return {"P1_pct": round(v[1], 1), "P2_pct": round(v[2], 1), "P3_pct": round(v[3], 1), "outcome": out}

b_rows = []
for norm in ("per_dish", "mean_profile"):
    t = R.groupby(keys).apply(lambda g: pd.Series(pct_outcome(g, AS_GIVEN, norm))).reset_index()
    t.insert(0, "norm", norm); b_rows.append(t)
b_tab = pd.concat(b_rows, ignore_index=True)
print("Main reading, both normalizations:")
print(b_tab[(b_tab.geom == "abs") & (b_tab.n_per_diag == 46) & (b_tab.center == "flank") & (b_tab.bg == "raw") & (b_tab.diag == "both")].to_string(index=False))
print("\nOutcomes over all", len(b_tab), "readings:"); print(b_tab.outcome.value_counts().to_string())
print("\nRange over readings (%):", {c: (b_tab[c].min(), b_tab[c].max()) for c in ["P1_pct", "P2_pct", "P3_pct"]})
print("Readings in which Packet 2 has the highest normalized center:", int((b_tab.P2_pct > b_tab[["P1_pct", "P3_pct"]].max(axis=1)).sum()), "of", len(b_tab))
print("Readings in which Packet 3 has the highest normalized center:", int((b_tab.P3_pct > b_tab[["P1_pct", "P2_pct"]].max(axis=1)).sum()), "of", len(b_tab))
print("\nPer-dish normalized center, main reading and background subtracted (%):")
print(R[main | mainbg].pivot_table(index=["packet", "dish"], columns="bg", values="pct").round(1).to_string())
pm = R[main]
print("\nTwo-sided t-test of per-dish % (main reading), Protocol 3 v Protocol 2:",
      ttest(pm[pm.packet == 3].pct, pm[pm.packet == 2].pct))

**Result: not reproduced.** Under the main reading the normalized centers are 81.3% (Packet 1),
82.8% (Packet 2) and 129.4% (Packet 3), where the claim has 74, 183 and 83%. Packet 3, not
Packet 2, has the highest normalized center in all 96 readings, and Packet 2 is 52.5 to 123.7%. Per
dish, the Protocol 3 files exceed the Protocol 2 files (t = 4.25, p = 0.008).

## Variant: Protocol 2 and Protocol 3 exchanged

The files labelled Protocol 3 (8 phases with 5-us delays) show the profile the paper reports for
Packet 2, and the reverse. This variant reruns the three criteria with Packet 2 read from the
Protocol 3 files and Packet 3 from the Protocol 2 files. It does not decide any outcome: the
outcomes above test the dataset as labelled.

In [ ]:
c_sw = R.groupby(keys).apply(lambda g: pd.Series(center_outcome(g, SWAPPED))).reset_index()
n_sw = R.groupby(keys).apply(lambda g: pd.Series(near_outcome(g, SWAPPED))).reset_index()
b_sw = pd.concat([R.groupby(keys).apply(lambda g: pd.Series(pct_outcome(g, SWAPPED, norm))).reset_index().assign(norm=norm)
                  for norm in ("per_dish", "mean_profile")], ignore_index=True)
for name, t in [("fig8a-equal-center-uptake", c_sw), ("fig8a-more-uptake-near-electrodes", n_sw), ("fig8b-packet2-best-targeting", b_sw)]:
    print(f"{name}, labels exchanged: outcomes over {len(t)} readings")
    print(t.groupby("bg").outcome.value_counts().to_string()); print()
sel = (n_sw.geom == "abs") & (n_sw.n_per_diag == 46) & (n_sw.center == "flank") & (n_sw.diag == "both")
print("Near electrodes, main geometry, labels exchanged:"); print(n_sw[sel].to_string(index=False))
sel = (b_sw.geom == "abs") & (b_sw.n_per_diag == 46) & (b_sw.center == "flank") & (b_sw.diag == "both")
print("\nNormalized center, main geometry, labels exchanged:"); print(b_sw[sel].to_string(index=False))
print("\nNormalized center over all readings, labels exchanged, by background (%):")
print(b_sw.groupby("bg")[["P1_pct", "P2_pct", "P3_pct"]].agg(["min", "max"]).round(1).to_string())

**Result: with the labels exchanged, the order of Figure 8 appears.** In the main geometry the
normalized centers are 70.2, 162.3 and 76.4% (Packets 1, 2, 3) with the background subtracted and
81.3, 129.4 and 82.8% raw, so fig8b-packet2-best-targeting would be partially reproduced in 92 of 96
readings and reproduced in 2. fig8a-more-uptake-near-electrodes would still not be reproduced in 43
of 48 readings (with the background subtracted, Packet 1 is 498.8 a.u., above 385, and Packet 3 over
Packet 2 gives p = 0.029, not below 0.01), and fig8a-equal-center-uptake in all 48, as Packet 1 is
unchanged. The data alone cannot say whether the files' labels or the paper's are exchanged; this
variant does not change the outcomes above.

## Summary

| Claim | Outcome |
|---|---|
| fig4-unipolar-strongest, fig4-bipolar-weaker, fig4-sham-no-uptake, fig4-twenty-fold-cancellation | Blocked, data missing |
| fig5a-trains-stronger-throughout, fig5a-packet-center-peak-e1-stronger, fig5bc-rotation-focuses-center | Blocked, data missing |
| fig6a-more-packets-more-uptake, fig6b-center-to-periphery-by-number, fig6-peak-narrower | Blocked, data missing |
| fig7a-amplitude-scales-uniformly, fig7bd-targeting-independent-of-amplitude, array-size-independence | Blocked, data missing |
| fig8a-equal-center-uptake | Not reproduced |
| fig8a-more-uptake-near-electrodes | Not reproduced |
| fig8b-packet2-best-targeting | Not reproduced |
| fig9-gap-fades-periphery-faster, fig9b-gap-boosts-targeting | Blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper measured ROIs by hand in MetaMorph on the
full-resolution images; here the electrodes are found automatically, ROIs are squares placed along
both diagonals of the reduced image (8 x 8 pixel means), and the scale of 0.65 um per pixel is
taken from the camera and objective, as the files give none. The ROI size, whether a background was
subtracted, which diagonal, and which ROI is the center are not given by the paper; each is run
every way, and no outcome depends on them. Reading (b) of the near-electrode claim, the ROI next to
e1, could not be run, as the files do not identify e1.

What was found about the dataset. It holds only the Figure 8 experiment (10 dishes), not the
single-cell time-lapse of Figure 4 or the monolayer experiments of Figures 5, 6, 7 and 9. The files
labelled Protocol 3 (8 phases with 5-us delays) show the strongest central focusing and the least
uptake near the electrodes, which the paper reports for Packet 2 (8 phases without delays), and the
files labelled Protocol 2 behave as the paper's Packet 3; either the labels or the paper's
assignment appear exchanged. The files give no pixel size (`resolution` 96, or 1.0 in dish 10), no
position or identity of the electrodes, and no exposure or gain, and the Packet 1 dishes, imaged on
another day, have about twice the background of the others. Dish 10 names no device. The species
summary names two species for one cell line, and the age (P1D) does not apply to a cell line.